# 5 1 content analysis

Peer-review copy. Inputs are pseudonymized processed data; see README.md and DATA_DICTIONARY.md. Original preparation/collection steps are outside this package.

In [ ]:
from review_setup import prepare_workspace
prepare_workspace()
from review_setup import initialize_stata
initialize_stata()


In [ ]:
import pandas as pd
from utils import StataDiD, get_source_corrections, fit_columns, SEED, get_coefs, vermilion, FONT_SIZE, LW, MZ

source_corrections = get_source_corrections(window_days=28).query('prior_corrections==0').reset_index(drop=True)
source_corrections = source_corrections.dropna(subset=fit_columns).reset_index(drop=True)
print(source_corrections.shape)
source_corrections.head()

### Propensity score matching

In [ ]:
# psm based on logistic regression
from sklearn.linear_model import LogisticRegression
from scipy.stats import zscore

source_corrections_zscore = source_corrections.dropna(subset=fit_columns).reset_index(drop=True)
print(source_corrections_zscore.shape)
for col in fit_columns:
    if col != 'user_is_verified':
        source_corrections_zscore[col] = zscore(source_corrections_zscore[col])

psm_model = LogisticRegression(C=1e6, random_state=SEED).fit(source_corrections_zscore[fit_columns], source_corrections_zscore['treated'])

In [ ]:
from causalml.match import NearestNeighborMatch

source_corrections_zscore = source_corrections_zscore.assign(propensity_score=psm_model.predict_proba(source_corrections_zscore[fit_columns])[:,1]).reset_index(drop=True)
psm = NearestNeighborMatch(replace=False, ratio=1, random_state=SEED, caliper=0.15)

# control group
control_matched = psm.match(data=source_corrections_zscore, treatment_col='treated', score_cols=['propensity_score']).reset_index(drop=True)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from utils import moderate_cyan, moderate_blue
plt.style.use("default")
plt.rcParams['font.size'] = FONT_SIZE

## visualization before psm
plt.subplots(layout="constrained")
sns.histplot(source_corrections_zscore['propensity_score'][source_corrections_zscore['treated']==0], label='No', element="step", color=moderate_cyan)
sns.histplot(source_corrections_zscore['propensity_score'][source_corrections_zscore['treated']==1], label='Yes', element="step", color=moderate_blue)
plt.xlabel('Propensity score')
plt.ylabel('Count')
plt.legend(title='Treated', loc='upper right')
plt.savefig('outputs/figures/propensity_score_correction_content.pdf')

In [ ]:
plt.subplots(layout="constrained")
sns.histplot(control_matched.query('treated==0')['propensity_score'], label='No', element="step", color=moderate_cyan)
sns.histplot(control_matched.query('treated==1')['propensity_score'], label='Yes', element="step", color=moderate_blue)
plt.xlabel('Propensity score')
plt.ylabel('Count')
plt.legend(title='Treated')
plt.savefig('outputs/figures/propensity_score_correction_content_matched.pdf')

In [ ]:
from scipy.stats import ttest_ind
def get_tresult(d1, d2):
    tresult = ttest_ind(d1, d2)
    p_text = '< 0.001' if tresult.pvalue < 0.001 else f'= {tresult.pvalue:.3f}'
    print(f't = {tresult.statistic:.3f} (df = {int(tresult.df):,}), p {p_text}')
get_tresult(source_corrections_zscore['propensity_score'][source_corrections_zscore['treated']==0], source_corrections_zscore['propensity_score'][source_corrections_zscore['treated']==1])

In [ ]:
get_tresult(control_matched.query('treated==0')['propensity_score'], control_matched.query('treated==1')['propensity_score'])

In [ ]:
source_corrections_zscore = source_corrections_zscore.merge(control_matched[['tweet_id']], on='tweet_id', how='inner')
source_corrections_zscore.to_parquet('outputs/data/source_corrections_content_matched.parquet')
print(source_corrections_zscore.shape)
# Check that regenerated matching reproduces the frozen membership used downstream.
expected_ids = set(pd.read_parquet("data/source_corrections_content_matched.parquet")["tweet_id"])
assert set(source_corrections_zscore["tweet_id"]) == expected_ids, "Matching differs from the supplied sample; check dependency versions."


In [ ]:
from utils import check_balance
balance = check_balance(source_corrections_zscore, fit_columns, tab_export=True, tab_label='covariate_balance_content_psm')

In [ ]:
balance.round(3)

In [ ]:
from utils import he_content_factors, prepare_content_count
from pystata import stata

for item in he_content_factors:
    fvar, fvar_name, user_var, filename, tcol, external_threshold = item[0], item[1], item[2], item[3], item[4], item[5]
    print('------------------------------')
    print(fvar_name)
    if fvar == 'misleadingness':
        post_counts = pd.read_parquet('data/ts_posts_daily.parquet.gzip').query('-4<=days_from_display<=4').reset_index(drop=True)
    else:
        post_counts = pd.read_parquet('data/ts_posts_weekly.parquet.gzip')

    post_counts_by_var = prepare_content_count(filename, fvar, tcol, external_threshold)
    post_counts = post_counts.merge(post_counts_by_var, on=['tweet_id', tcol], how='left')
    post_counts[f'tweet_count_{fvar}_yes'] = post_counts[f'tweet_count_{fvar}_yes'].fillna(0)
    post_counts[f'tweet_count_{fvar}_no'] = post_counts[f'tweet_count_{fvar}_no'].fillna(0)
    post_counts[f'{fvar}_ratio'] = post_counts[f'tweet_count_{fvar}_yes']/post_counts['tweet_count']
    post_counts = post_counts.merge(source_corrections_zscore, on='tweet_id', how='inner')
    stata_did = StataDiD(post_counts, 'treated', 'tweet_id', tcol, -4, 4)
    print('[Leads-and-lags counts]', end=' ')
    stata_did.multi_did(reg=f'qui ppmlhdfe', dev=f'tweet_count_{fvar}_yes', random_unit=False, cluster='account_cluster')
    _, _, _, _, _, _ = get_coefs(f'qui ppmlhdfe', 8, 15, stat_type='z', irr_or=1, unit_group=True, full_return=True)
    if fvar == 'negative':
        stata.run('testparm treated_periods_pre*')
    print('[Pre-post counts]', end=' ')
    stata_did.tw_did(reg=f'qui ppmlhdfe', dev=f'tweet_count_{fvar}_yes', random_unit=False, cluster='account_cluster')
    mean_external, std_err_external, stat_external, pval_external, cll_external, chl_external = get_coefs(f'qui ppmlhdfe', 2, 3, stat_type='z', irr_or=1, unit_group=True, full_return=True)
    print('[Ratio]', end=' ')
    stata_did.tw_did(reg=f'qui xtreg', dev=f'{fvar}_ratio', random_unit=False, cluster='account_cluster', condition=f'if tweet_count>0 & ({fvar}_ratio>=0 & {fvar}_ratio<=1)')
    _, _, _, _, _, _ = get_coefs(f'qui xtreg', 2, 3, stat_type='t', unit_group=True, full_return=True)